# ConvNeXtV2-Tiny — Baseline Stage 1 (RQ1)

Notebook ini membangun **baseline ConvNeXtV2-Tiny** untuk klasifikasi *Diabetic Retinopathy* 5 kelas, independen dari pipeline CAD-ViT.

**Catatan penting tentang file asli.** File yang diberi nama `ConvNeXtV2_Tiny.ipynb` yang menjadi dasar notebook ini ternyata, di luar judulnya, berisi pipeline **CAD-ViT** secara penuh: model `CADViTCandidate` (MFB, Coordinate Attention, IRB/EMB blocks), konfigurasi `ARCH_CONFIG`/`ABLATION_CONFIGS`, serta fungsi `run_experiment`/`run_experiment_v2` yang semuanya mereproduksi arsitektur paper CAD-ViT (DOI 10.1109/ACCESS.2025.3572455) — bukan ConvNeXtV2-Tiny. Hanya cell judul yang menyebut "ConvNeXtV2-Tiny".

Notebook ini **mengganti seluruh bagian model/training CAD-ViT** tersebut dengan ConvNeXtV2-Tiny yang sesungguhnya (via `timm`), sambil **mempertahankan infrastruktur yang arsitektur-agnostik** dari file asal — validasi struktur dataset resmi, pembacaan label APTOS/DDR, split 8:1:1 ter-stratifikasi dengan SHA-256 grouping (seed 42) — dan memindahkannya ke direktori independen `ConvNeXtV2-DR/` sesuai struktur direktori pada project brief, agar manifest/checkpoint ConvNeXtV2-Tiny **tidak pernah menimpa** manifest/checkpoint CAD-ViT.

Dataset resmi (sama dengan yang dipakai CAD-ViT, dibagi sebagai raw image, bukan manifest):
- APTOS 2019: https://www.kaggle.com/competitions/aptos2019-blindness-detection/data
- OIA-DDR: https://github.com/nkicsl/DDR-dataset

**Scope notebook ini — Stage 1: Baseline Training (RQ1), sesuai §3 dan Stage 1 §5 pada project brief:**

| Komponen | Baseline (wajib sesuai brief) |
| --- | --- |
| Arsitektur | ConvNeXtV2-Tiny (timm), BUKAN ConvNeXt-Tiny atau CAD-ViT |
| Bobot | ImageNet pretrained |
| Head | 5 kelas |
| Input | RGB 3 kanal |
| Preprocessing tambahan | Tidak ada — hanya resize + normalisasi ImageNet standar model pretrained |
| Augmentasi | Nonaktif |
| Class balancing | Nonaktif |
| Loss | CrossEntropyLoss standar |
| Precision training awal | FP32 |
| Hardware | GPU (idealnya NVIDIA A100) |
| Evaluasi | Validation dan test |

Stage 2 (augmentasi/balancing/severity-aware learning — RQ1), Stage 3 (CPU inference benchmarking — RQ2), dan Stage 4 (precision FP16/INT8 — RQ3) menyusul **setelah** baseline ini berjalan dan tervalidasi, sesuai prioritas pengembangan pada brief.


### 1.1 Instalasi dan import

In [1]:
# Jalankan sekali per sesi Colab/runtime. timm menyediakan ConvNeXtV2-Tiny dan transform standarnya.
!pip install -q timm

In [2]:
import os
import json
import random
from dataclasses import dataclass, asdict
from hashlib import sha256
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import timm
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFile, ImageOps

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    cohen_kappa_score,
)

ImageFile.LOAD_TRUNCATED_IMAGES = False
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
SPLIT_SEED = 42  # sama dengan seed split CAD-ViT agar perbandingan arsitektur tetap mungkin (lihat §2 di bawah)

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
print("timm:", timm.__version__)


torch: 2.14.1+cu130 | cuda available: True
timm: 1.0.31


### 1.2 Direktori independen `ConvNeXtV2-DR/`

Sesuai diagram direktori pada brief:

```
Google Drive/
├── CAD_ViT/
│   ├── datasets/        <- raw image APTOS2019/ dan DDR/ (DIBAGI, dibaca read-only oleh notebook ini)
│   ├── manifests/       <- TIDAK PERNAH disentuh notebook ini
│   ├── outputs/
│   └── checkpoints/
│
└── ConvNeXtV2-DR/       <- SELURUH output notebook ini (manifests/outputs/checkpoints/benchmarks)
    ├── manifests/
    ├── outputs/
    ├── checkpoints/
    └── benchmarks/
```

`ConvNeXtV2-DR/` tidak memiliki subfolder `datasets/` sendiri — gambar mentah dibagi dengan CAD-ViT (dataset resmi yang sama), tetapi manifest, output, checkpoint, dan benchmark sepenuhnya independen. Guard di bawah memastikan root ConvNeXtV2-Tiny tidak pernah berada di dalam `CAD_ViT/` dan tidak pernah sama dengan direktori dataset.

In [3]:
# Mount Google Drive (Colab). Dataset mentah dibagi dengan CAD-ViT; manifest/output/checkpoint tidak.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Menjalankan di luar Colab (Drive tidak di-mount otomatis).")

os.environ.setdefault("CONVNEXTV2_DATA_ROOT", "/content/drive/MyDrive/CAD_ViT/datasets")
os.environ.setdefault("CONVNEXTV2_RUN_ROOT", "/content/drive/MyDrive/ConvNeXtV2-DR")

DATA_ROOT = Path(os.environ["CONVNEXTV2_DATA_ROOT"]).expanduser().resolve()
RUN_ROOT = Path(os.environ["CONVNEXTV2_RUN_ROOT"]).expanduser().resolve()
APTOS_ROOT = DATA_ROOT / "APTOS2019"
DDR_ROOT = DATA_ROOT / "DDR"

# Guard: root ConvNeXtV2-Tiny tidak boleh bertumpang tindih dengan direktori CAD_ViT atau dataset mentah.
if "CAD_VIT" in [p.upper() for p in RUN_ROOT.parts] or RUN_ROOT == DATA_ROOT or DATA_ROOT in RUN_ROOT.parents or RUN_ROOT in DATA_ROOT.parents:
    raise ValueError(
        "CONVNEXTV2_RUN_ROOT tidak boleh berada di dalam / bertumpang tindih dengan direktori CAD_ViT "
        "atau direktori dataset mentah. Manifest/checkpoint ConvNeXtV2-Tiny harus independen."
    )

for sub in ("manifests", "outputs", "checkpoints", "benchmarks"):
    (RUN_ROOT / sub).mkdir(parents=True, exist_ok=True)

print("DATA_ROOT (raw images, dibagi dengan CAD-ViT, read-only):", DATA_ROOT)
print("RUN_ROOT  (ConvNeXtV2-Tiny saja, read-write):            ", RUN_ROOT)


Mounted at /content/drive
DATA_ROOT (raw images, dibagi dengan CAD-ViT, read-only): /content/drive/MyDrive/CAD_ViT/datasets
RUN_ROOT  (ConvNeXtV2-Tiny saja, read-write):             /content/drive/MyDrive/ConvNeXtV2-DR


### 1.3 Validasi struktur dataset resmi

In [4]:
def find_one(root, name, directory=False):
    matches = sorted(p for p in root.rglob(name) if p.is_dir() == directory)
    if len(matches) != 1:
        kind = "direktori" if directory else "file"
        raise FileNotFoundError(f"Harus ada tepat satu {kind} {name} di {root}; ditemukan {len(matches)}")
    return matches[0]

if not APTOS_ROOT.is_dir():
    raise FileNotFoundError(f"Folder APTOS2019 tidak ditemukan: {APTOS_ROOT}")
APTOS_TRAIN_CSV = find_one(APTOS_ROOT, "train.csv")
APTOS_IMAGE_DIR = find_one(APTOS_ROOT, "train_images", directory=True)
print("APTOS2019 siap:", APTOS_TRAIN_CSV, "|", APTOS_IMAGE_DIR)

DDR_STATUS = "folder DDR belum tersedia"
if DDR_ROOT.is_dir():
    DDR_STATUS = "folder DDR tersedia"
print("Status DDR:", DDR_STATUS, "-", DDR_ROOT)


APTOS2019 siap: /content/drive/MyDrive/CAD_ViT/datasets/APTOS2019/train.csv | /content/drive/MyDrive/CAD_ViT/datasets/APTOS2019/train_images
Status DDR: folder DDR tersedia - /content/drive/MyDrive/CAD_ViT/datasets/DDR


## 2. Manifest ConvNeXtV2-Tiny — independen dari CAD-ViT

Logika split di bawah (grouping SHA-256, split 8:1:1 ter-stratifikasi per kelas, seed 42) **identik secara metodologis** dengan yang dipakai CAD-ViT, karena logika ini arsitektur-agnostik — ini memenuhi brief §4: *"Use identical image IDs and split assignments to CAD-ViT when conducting controlled architecture comparisons, where feasible."* Namun manifest **ditulis ke file dan direktori terpisah** (`RUN_ROOT/manifests/`), dengan fingerprint SHA-256 sendiri, dan **tidak pernah menimpa manifest lama secara diam-diam** (bila manifest sudah ada dan isinya berbeda, `prepare_manifest` akan berhenti dengan error, bukan menimpa).

`compare_with_other_manifest` di bawah bersifat opsional: ia **melaporkan** selisih image_id/split terhadap manifest CAD-ViT tanpa memaksakan kecocokan, sesuai brief: *"If dataset versions or image counts differ, investigate the differences rather than forcing the manifests to match."*

In [5]:
def _sha256_file(path):
    digest = sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def validate_records(records, dataset_root, allow_label_conflicts=False):
    '''Kolom wajib: image_id, label, image_path (relatif terhadap dataset_root).'''
    needed = {"image_id", "label", "image_path"}
    if not needed.issubset(records.columns):
        raise ValueError(f"Kolom wajib: {sorted(needed)}")
    result = records.copy()
    result["image_id"] = result["image_id"].astype("string").str.strip()
    if result["image_id"].isna().any() or result["image_id"].eq("").any():
        raise ValueError("image_id kosong")
    if result["image_id"].duplicated().any():
        raise ValueError("Duplikasi image_id; periksa pasangan gambar-label")
    labels = pd.to_numeric(result["label"], errors="coerce")
    if (~labels.isin(range(5))).any():
        raise ValueError("Label harus integer 0..4")
    result["label"] = labels.astype(int)
    root = Path(dataset_root).resolve()
    hashes, paths = [], []
    for raw in result["image_path"]:
        if pd.isna(raw):
            raise ValueError("image_path kosong")
        relative = Path(str(raw))
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"Path gambar harus relatif dan aman: {raw}")
        full = (root / relative).resolve()
        if root not in full.parents or not full.is_file() or full.suffix.lower() not in IMAGE_EXTENSIONS:
            raise FileNotFoundError(f"Gambar hilang/di luar root/ekstensi tidak didukung: {raw}")
        with Image.open(full) as im:
            im.verify()
        with Image.open(full) as im:
            im.load()
            if min(im.size) < 1:
                raise ValueError(f"Dimensi tidak valid: {raw}")
        paths.append(full.relative_to(root).as_posix())
        hashes.append(_sha256_file(full))
    result["image_path"] = paths
    result["sha256"] = hashes
    conflicts = result.groupby("sha256")["label"].nunique()
    if (conflicts > 1).any() and not allow_label_conflicts:
        raise ValueError("Gambar identik memiliki label berbeda; perlu audit manual")
    result["group_id"] = result["sha256"].map(lambda h: "hash:" + h)
    return result[["image_id", "label", "image_path", "group_id", "sha256"]]


def aptos_records():
    '''APTOS official train.csv + train_images/, dibaca langsung (tanpa EDA sebelumnya).'''
    frame = pd.read_csv(APTOS_TRAIN_CSV)
    frame = frame.rename(columns={"id_code": "image_id", "diagnosis": "label"})
    frame["image_path"] = frame["image_id"].map(lambda i: f"train_images/{i}.png")
    return frame[["image_id", "label", "image_path"]]


def ddr_records():
    '''Baca anotasi resmi DDR DR_grading (train/valid/test.txt); simpan official_split untuk audit.'''
    if not DDR_ROOT.is_dir():
        raise FileNotFoundError(DDR_ROOT)
    direct = DDR_ROOT / "DDR-dataset" / "DR_grading"
    candidates = [direct] if direct.is_dir() else sorted(
        p for p in DDR_ROOT.rglob("DR_grading") if p.is_dir()
        and all((p / name).is_file() for name in ("train.txt", "valid.txt", "test.txt"))
    )
    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Harus ada tepat satu DR_grading dengan train/valid/test.txt di {DDR_ROOT}; "
            f"ditemukan {len(candidates)}"
        )
    grading_root = candidates[0]
    images = {}
    for image_path in grading_root.rglob("*"):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            if image_path.name in images:
                raise ValueError(f"Nama gambar DDR ambigu: {image_path.name}")
            images[image_path.name] = image_path
    rows, seen_names = [], set()
    for official_split, annotation_name in (("train", "train.txt"), ("valid", "valid.txt"), ("test", "test.txt")):
        annotation = grading_root / annotation_name
        if not annotation.is_file():
            raise FileNotFoundError(annotation)
        with annotation.open("r", encoding="utf-8-sig") as stream:
            for line_number, line in enumerate(stream, 1):
                if not line.strip():
                    continue
                fields = line.split()
                if len(fields) != 2:
                    raise ValueError(f"Format anotasi DDR salah: {annotation}:{line_number}")
                filename, raw_label = fields
                if filename != Path(filename).name or "/" in filename or chr(92) in filename:
                    raise ValueError(f"Nama gambar tidak aman: {filename}")
                if filename in seen_names:
                    raise ValueError(f"Nama anotasi DDR duplikat: {filename}")
                seen_names.add(filename)
                try:
                    label = int(raw_label)
                except ValueError as exc:
                    raise ValueError(f"Label DDR tidak numerik: {annotation}:{line_number}") from exc
                if raw_label != str(label) or label not in range(6):
                    raise ValueError(f"Label DDR di luar 0-5: {annotation}:{line_number}")
                image_path = images.get(filename)
                if image_path is None:
                    raise FileNotFoundError(f"Gambar DDR untuk anotasi hilang: {filename}")
                rows.append({
                    "image_id": filename, "label": label,
                    "image_path": image_path.resolve().relative_to(DDR_ROOT.resolve()).as_posix(),
                    "official_split": official_split,
                })
    all_rows = pd.DataFrame(rows)
    grading = all_rows[all_rows["label"].isin(range(5))].copy()
    records = grading[["image_id", "label", "image_path", "official_split"]].copy()
    records.attrs["ddr_audit"] = {
        "annotation_root": grading_root.resolve().relative_to(DDR_ROOT.resolve()).as_posix(),
        "all_annotations": len(all_rows),
        "ungradable_label_5": int((all_rows["label"] == 5).sum()),
        "grading_labels_0_4": len(grading),
        "official_split_grading": grading["official_split"].value_counts().to_dict(),
    }
    return records


def _group_split_8_1_1(frame, seed):
    '''SHA-256 grouping + split 8:1:1 ter-stratifikasi per kelas, tanpa exact-duplicate leakage.
    Identik secara metodologis dengan splitter CAD-ViT (seed 42), agar perbandingan arsitektur
    tetap mungkin, tetapi dijalankan di sini secara independen pada manifest ConvNeXtV2-Tiny.'''
    from sklearn.model_selection import train_test_split

    frame = frame.copy()
    groups = frame.groupby("group_id", sort=True).agg(
        stratum=("label", "max"), images=("label", "size")
    ).reset_index()
    strata = groups["stratum"].value_counts().reindex(range(5), fill_value=0)
    if (strata < 10).any():
        raise ValueError(f"Terlalu sedikit grup per kelas: {strata.to_dict()}")
    train_g, held_g = train_test_split(groups, test_size=0.2, random_state=seed, stratify=groups["stratum"])
    val_g, test_g = train_test_split(held_g, test_size=0.5, random_state=seed, stratify=held_g["stratum"])
    splits = ("train", "val", "test")
    allocation = {str(g): part for part, subset in
                  (("train", train_g), ("val", val_g), ("test", test_g)) for g in subset["group_id"]}
    n_images = len(frame)
    targets = {"train": round(0.8 * n_images), "val": round(0.1 * n_images)}
    targets["test"] = n_images - targets["train"] - targets["val"]
    ratios = {"train": 0.8, "val": 0.1, "test": 0.1}
    by_group = {str(g): rows for g, rows in frame.groupby("group_id", sort=True)}
    sizes = {part: 0 for part in splits}
    class_counts = {part: [0] * 5 for part in splits}
    totals = frame["label"].value_counts().reindex(range(5), fill_value=0).to_list()
    for group_id, rows in by_group.items():
        part = allocation[group_id]
        sizes[part] += len(rows)
        for label, count in rows["label"].value_counts().items():
            class_counts[part][int(label)] += int(count)

    for _ in range(n_images + 1):
        excess = {part: sizes[part] - targets[part] for part in splits}
        if all(value == 0 for value in excess.values()):
            break
        donors = [part for part in splits if excess[part] > 0]
        receivers = [part for part in splits if excess[part] < 0]
        if not donors or not receivers:
            raise RuntimeError("Rasio 8:1:1 tidak dapat ditepatkan")
        donor = sorted(donors, key=lambda part: (-excess[part], splits.index(part)))[0]
        receiver = sorted(receivers, key=lambda part: (excess[part], splits.index(part)))[0]
        choices = []
        for group_id, rows in by_group.items():
            if allocation[group_id] != donor or len(rows) != 1:
                continue
            label = int(rows["label"].iloc[0])
            if class_counts[donor][label] <= 1:
                continue
            before = ((class_counts[donor][label] - ratios[donor] * totals[label]) ** 2
                      + (class_counts[receiver][label] - ratios[receiver] * totals[label]) ** 2)
            after = ((class_counts[donor][label] - 1 - ratios[donor] * totals[label]) ** 2
                     + (class_counts[receiver][label] + 1 - ratios[receiver] * totals[label]) ** 2)
            tie = sha256(f"{seed}:{group_id}".encode()).hexdigest()
            choices.append((after - before, tie, group_id, label))
        if not choices:
            raise RuntimeError("Tidak ada grup tunggal yang dapat dipindah tanpa leakage")
        _, _, group_id, label = min(choices)
        allocation[group_id] = receiver
        sizes[donor] -= 1
        sizes[receiver] += 1
        class_counts[donor][label] -= 1
        class_counts[receiver][label] += 1
    else:
        raise RuntimeError("Penyesuaian rasio tidak konvergen")

    for _ in range(15):
        missing = [(part, label) for part in splits for label in range(5) if class_counts[part][label] == 0]
        if not missing:
            break
        receiver, label = missing[0]
        swaps = []
        for donor in splits:
            if donor == receiver or class_counts[donor][label] <= 1:
                continue
            for source_id, source_rows in by_group.items():
                if allocation[source_id] != donor or len(source_rows) != 1 or int(source_rows["label"].iloc[0]) != label:
                    continue
                for return_id, return_rows in by_group.items():
                    if allocation[return_id] != receiver or len(return_rows) != 1:
                        continue
                    return_label = int(return_rows["label"].iloc[0])
                    if class_counts[receiver][return_label] <= 1:
                        continue
                    tie = sha256(f"{seed}:{source_id}:{return_id}".encode()).hexdigest()
                    swaps.append((tie, donor, source_id, return_id, return_label))
        if not swaps:
            raise RuntimeError("Semua kelas tidak dapat diwakili tanpa memecah grup")
        _, donor, source_id, return_id, return_label = min(swaps)
        allocation[source_id], allocation[return_id] = receiver, donor
        class_counts[donor][label] -= 1
        class_counts[receiver][label] += 1
        class_counts[receiver][return_label] -= 1
        class_counts[donor][return_label] += 1
    else:
        raise RuntimeError("Penyesuaian kelas tidak konvergen")

    frame["split"] = frame["group_id"].astype(str).map(allocation)
    observed = frame["split"].value_counts().to_dict()
    if len(frame) != n_images or frame["split"].isna().any():
        raise AssertionError("Ada gambar hilang dari split")
    if any(observed.get(part, 0) != targets[part] for part in splits):
        raise AssertionError(f"Rasio tidak sesuai: {observed} vs {targets}")
    if frame.groupby("group_id")["split"].nunique().max() != 1:
        raise AssertionError("Group leakage")
    if frame.groupby("sha256")["split"].nunique().max() != 1:
        raise AssertionError("Exact-duplicate leakage")
    return frame


def prepare_manifest(dataset, seed=SPLIT_SEED):
    '''Tulis manifest ConvNeXtV2-Tiny ke RUN_ROOT/manifests/. Tidak pernah menimpa manifest
    berbeda secara diam-diam, dan tidak pernah membaca/menulis direktori CAD-ViT.'''
    if dataset not in {"APTOS2019", "DDR"}:
        raise ValueError("dataset harus APTOS2019 atau DDR")
    root = APTOS_ROOT if dataset == "APTOS2019" else DDR_ROOT
    raw = aptos_records() if dataset == "APTOS2019" else ddr_records()
    frame = validate_records(raw, root, allow_label_conflicts=(dataset == "APTOS2019"))
    if dataset == "DDR":
        frame["official_split"] = frame["image_id"].map(raw.set_index("image_id")["official_split"])

    frame = _group_split_8_1_1(frame, seed)
    frame = frame.sort_values("image_id").reset_index(drop=True)
    fingerprint = sha256(frame.to_csv(index=False).encode("utf-8")).hexdigest()

    out = RUN_ROOT / "manifests"
    manifest_path = out / f"{dataset}_convnextv2tiny_seed{seed}.csv"
    meta_path = out / f"{dataset}_convnextv2tiny_seed{seed}.json"
    if manifest_path.exists() or meta_path.exists():
        if not (manifest_path.exists() and meta_path.exists()):
            raise RuntimeError("Manifest atau metadata tidak lengkap; periksa manual")
        old = pd.read_csv(manifest_path, dtype={"image_id": "string", "group_id": "string"})
        old_fingerprint = sha256(old.to_csv(index=False).encode("utf-8")).hexdigest()
        if fingerprint != old_fingerprint:
            raise RuntimeError("Manifest lama berbeda; jangan timpa split eksperimen yang sudah ada")
    else:
        frame.to_csv(manifest_path, index=False)
        meta = {
            "dataset": dataset, "seed": seed, "manifest_sha256": fingerprint,
            "manifest_file_sha256": _sha256_file(manifest_path),
            "architecture": "ConvNeXtV2-Tiny", "grouping": "exact SHA-256",
            "split_ratio": "8:1:1 image-exact group-stratified",
            "preprocess": "none baked into manifest; standard ImageNet transform applied at load time",
            "root": str(root), "counts": frame.groupby(["split", "label"]).size().to_dict(),
        }
        meta["counts"] = {f"{k[0]}:{k[1]}": int(v) for k, v in meta["counts"].items()}
        meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

    print("Dataset:", dataset, "| total gambar:", len(frame), "| grup SHA-256:", frame["sha256"].nunique())
    print("Jumlah gambar per split:", frame["split"].value_counts().reindex(["train", "val", "test"]).to_dict())
    print("Exact-duplicate leakage:", int((frame.groupby("sha256")["split"].nunique() > 1).sum()))
    print("Manifest:", manifest_path)
    return manifest_path


def compare_with_other_manifest(manifest_path, other_manifest_path):
    '''Opsional (brief SS4): bandingkan image_id/split dengan manifest CAD-ViT TANPA memaksakan
    kecocokan. Jika dataset/jumlah gambar berbeda, selidiki -- jangan dipaksa sama.'''
    mine = pd.read_csv(manifest_path, dtype={"image_id": "string"})
    other = pd.read_csv(other_manifest_path, dtype={"image_id": "string"})
    common = set(mine.image_id) & set(other.image_id)
    merged = mine.merge(other, on="image_id", suffixes=("_v2tiny", "_other"))
    mismatched = merged[merged["split_v2tiny"] != merged["split_other"]] if "split_other" in merged else merged.iloc[0:0]
    report = {
        "n_mine": len(mine), "n_other": len(other), "n_common": len(common),
        "n_only_mine": len(set(mine.image_id) - common),
        "n_only_other": len(set(other.image_id) - common),
        "n_split_mismatch_on_common": len(mismatched),
    }
    print(json.dumps(report, indent=2))
    return report


## 3. Dataset & DataLoader — preprocessing baseline ConvNeXtV2-Tiny (TANPA preprocessing retina)

Sesuai brief §3: baseline **tidak** menggunakan crop border hitam, circular mask, Wiener, CLAHE, green-channel extraction, Laplacian enhancement, atau enhancement retina lain apa pun. Satu-satunya transformasi input yang dipakai adalah transformasi standar yang dipersyaratkan oleh bobot pretrained ConvNeXtV2-Tiny itu sendiri: konversi RGB, resize ke resolusi native model, dan normalisasi ImageNet — diambil langsung dari `timm.data.resolve_data_config`, bukan ditulis manual, supaya konsisten dengan bobot pretrained yang dipakai.

Augmentasi dan class balancing dinonaktifkan pada baseline ini (brief §3); keduanya diaktifkan kembali pada Stage 2.

In [6]:
MODEL_NAME = "convnextv2_tiny.fcmae_ft_in22k_in1k"  # ConvNeXtV2-Tiny, ImageNet-1k fine-tuned, 224px
if "convnextv2" not in MODEL_NAME.lower():
    raise ValueError(
        "MODEL_NAME harus varian ConvNeXtV2 -- JANGAN keliru dengan ConvNeXt-Tiny atau CAD-ViT (brief SS1)."
    )

_probe_model = timm.create_model(MODEL_NAME, pretrained=False)
_data_cfg = timm.data.resolve_data_config({}, model=_probe_model)
IMAGE_SIZE = _data_cfg["input_size"][-1]
del _probe_model

BASELINE_TRANSFORM = timm.data.create_transform(**_data_cfg, is_training=False)
# is_training=False -> resize/crop ke IMAGE_SIZE + ToTensor + Normalize(ImageNet mean/std) SAJA.
# Tidak ada random flip/rotation/shear (augmentasi nonaktif pada baseline) dan tidak ada preprocessing retina.

print("Resolusi input model:", IMAGE_SIZE)
print("Transform baseline:", BASELINE_TRANSFORM)


class FundusDatasetV2Tiny(Dataset):
    '''Stage 1 baseline: RGB convert -> transform standar model pretrained saja.'''

    def __init__(self, frame, root, transform=BASELINE_TRANSFORM):
        self.frame = frame.reset_index(drop=True)
        self.root = Path(root)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(self.root / row.image_path) as image:
            rgb = ImageOps.exif_transpose(image).convert("RGB")
        return self.transform(rgb), int(row.label)


def _seed_worker(worker_id):
    seed = torch.initial_seed() % (2**32)
    np.random.seed(seed)
    random.seed(seed)


def make_loaders_baseline(manifest_path, dataset, batch_size=32, seed=SPLIT_SEED, num_workers=2):
    '''balance=False (shuffle biasa), augment=False (transform eval-only untuk SEMUA split),
    sesuai baseline brief SS3.'''
    meta_path = Path(manifest_path).with_suffix(".json")
    if not meta_path.is_file():
        raise FileNotFoundError(f"Metadata manifest hilang: {meta_path}")
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    if meta.get("dataset") != dataset or meta.get("manifest_file_sha256") != _sha256_file(manifest_path):
        raise ValueError("Dataset atau hash manifest tidak cocok dengan metadata")
    frame = pd.read_csv(manifest_path, dtype={"image_id": "string", "group_id": "string"})
    if set(frame.split) != {"train", "val", "test"}:
        raise ValueError("Manifest tidak lengkap")
    root = APTOS_ROOT if dataset == "APTOS2019" else DDR_ROOT
    generator = torch.Generator().manual_seed(seed)
    loaders = {}
    for split in ("train", "val", "test"):
        subset = frame[frame.split == split]
        ds = FundusDatasetV2Tiny(subset, root)
        loaders[split] = DataLoader(
            ds, batch_size=batch_size, shuffle=(split == "train"), num_workers=num_workers,
            pin_memory=torch.cuda.is_available(), worker_init_fn=_seed_worker,
            generator=generator, persistent_workers=num_workers > 0,
        )
    return loaders


Resolusi input model: 224
Transform baseline: Compose(
    Resize(size=256, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)


## 4. Model — ConvNeXtV2-Tiny

`build_model` memuat ConvNeXtV2-Tiny dari `timm` dengan bobot ImageNet pretrained dan head 5 kelas, sesuai brief §3. Sel di bawah menjalankan sanity-check forward+backward pada CPU dengan bobot acak (bukan pretrained) sebelum training sungguhan, agar kesalahan shape terdeteksi lebih awal tanpa biaya GPU.

In [7]:
def build_model(num_classes=5, pretrained=True):
    model = timm.create_model(MODEL_NAME, pretrained=pretrained, num_classes=num_classes)
    return model


_sanity_model = build_model(pretrained=False)
_sanity_model.train()
_dummy = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, requires_grad=True)
_logits = _sanity_model(_dummy)
assert _logits.shape == (2, 5), _logits.shape
_logits.square().mean().backward()
_missing_grad = [n for n, p in _sanity_model.named_parameters() if p.requires_grad and p.grad is None]
assert not _missing_grad, _missing_grad
print("Sanity check OK -- logits shape:", tuple(_logits.shape))
print("Total parameter ConvNeXtV2-Tiny:", f"{sum(p.numel() for p in _sanity_model.parameters()):,}")
del _sanity_model, _dummy, _logits


Sanity check OK -- logits shape: (2, 5)
Total parameter ConvNeXtV2-Tiny: 27,870,341


## 5. Konfigurasi baseline Stage 1

Field yang ditandai **EXPLICIT (brief SS3)** di bawah tidak boleh diubah untuk run baseline ini -- `run_baseline` menolak menjalankan bila diubah. Field lain (`epochs`, `learning_rate`, `weight_decay`, `batch_size`) **tidak ditetapkan oleh brief** untuk ConvNeXtV2-Tiny (brief hanya memberi konfigurasi baseline arsitektur/precision/augmentasi/balancing/loss, bukan hyperparameter optimizer) -- nilai di bawah adalah *implementation assumption* yang wajar untuk fine-tuning backbone pretrained, dan harus dicatat sebagai asumsi implementasi pada laporan, sama seperti notebook CAD-ViT asal menandai pilihan non-eksplisit.

In [8]:
@dataclass
class BaselineConfig:
    dataset: str = "APTOS2019"       # "APTOS2019" atau "DDR"
    seed: int = SPLIT_SEED
    epochs: int = 50                 # implementation assumption -- brief tidak menetapkan jumlah epoch baseline
    learning_rate: float = 5e-5      # implementation assumption -- lr kecil wajar untuk fine-tune backbone pretrained
    weight_decay: float = 0.05
    batch_size: int = 32             # implementation assumption
    num_workers: int = 2
    precision: str = "fp32"          # EXPLICIT (brief SS3): training precision awal = FP32
    augment: bool = False            # EXPLICIT (brief SS3): augmentasi nonaktif pada baseline
    balance: bool = False            # EXPLICIT (brief SS3): class balancing nonaktif pada baseline
    wandb_mode: str = "online"      # "online" | "offline" | "disabled"


## 6. Training & evaluasi

Metrik mengikuti brief SS6: accuracy, precision/recall/F1 macro, F1 dan recall per kelas, dan Quadratic Weighted Kappa (QWK). `run_baseline` menyimpan: checkpoint terbaik (`best.pt`, dipilih berdasarkan validation macro-F1) dan terakhir (`last.pt`), `training_history.csv`, metrik val/test (`*_metrics.json`), prediksi test (`*_predictions.csv`), confusion matrix (`*_confusion_matrix.csv`), classification report (`*_classification_report.txt`), dan konfigurasi eksperimen (`config.json`) -- seluruhnya di bawah `RUN_ROOT/checkpoints/<run_name>/`.

**Catatan W&B (brief SS8 butir 12).** Berhasilnya `wandb.init()`/autentikasi TIDAK berarti metrik benar-benar ter-log; periksa dashboard W&B secara langsung setelah run untuk memverifikasi bahwa step/metric benar-benar tercatat, terutama bila `wandb_mode="offline"` (perlu `wandb sync` manual) atau koneksi jaringan tidak stabil.

In [9]:
DR_LABELS = {0: "No DR", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferative DR"}


def set_experiment_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def compute_metrics(y_true, y_pred):
    labels = list(range(5))
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="macro", zero_division=0)
    per_class_precision, per_class_recall, per_class_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average=None, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    qwk = cohen_kappa_score(y_true, y_pred, labels=labels, weights="quadratic")
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "qwk": float(qwk),
        "per_class_f1": {DR_LABELS[k]: float(v) for k, v in enumerate(per_class_f1)},
        "per_class_recall": {DR_LABELS[k]: float(v) for k, v in enumerate(per_class_recall)},
        "per_class_precision": {DR_LABELS[k]: float(v) for k, v in enumerate(per_class_precision)},
        "confusion_matrix": cm.tolist(),
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    true, pred, loss_sum, n = [], [], 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss_sum += criterion(logits, y).item() * len(y)
        n += len(y)
        true.extend(y.cpu().tolist())
        pred.extend(logits.argmax(1).cpu().tolist())
    metrics = compute_metrics(true, pred)
    metrics["loss"] = loss_sum / n
    metrics["n"] = n
    metrics["y_true"] = true
    metrics["y_pred"] = pred
    return metrics


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        n += len(y)
    return total_loss / n


def _save_checkpoint(path, state):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    torch.save(state, temp)
    os.replace(temp, path)


def run_baseline(config: BaselineConfig, manifest_path, run_name):
    if config.dataset not in {"APTOS2019", "DDR"}:
        raise ValueError("dataset harus APTOS2019 atau DDR")
    if config.precision != "fp32":
        raise ValueError("Stage 1 baseline wajib FP32 (brief SS3); precision lain ada di Stage 4 / RQ3")
    if config.augment or config.balance:
        raise ValueError("Stage 1 baseline wajib augmentasi & balancing nonaktif (brief SS3)")
    if not run_name or Path(run_name).name != run_name:
        raise ValueError("run_name harus satu nama direktori aman")

    set_experiment_seed(config.seed)
    manifest_path = Path(manifest_path)
    manifest_hash = _sha256_file(manifest_path)

    run_dir = RUN_ROOT / "checkpoints" / run_name
    if run_dir.exists():
        raise FileExistsError(f"Run sudah ada; pakai nama baru atau hapus manual: {run_dir}")
    run_dir.mkdir(parents=True)

    loaders = make_loaders_baseline(manifest_path, config.dataset, config.batch_size,
                                     config.seed, config.num_workers)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        print("PERINGATAN: GPU tidak tersedia; brief merekomendasikan A100 untuk training baseline.")

    model = build_model(num_classes=5, pretrained=True).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    criterion = nn.CrossEntropyLoss()

    record = {
        "architecture": MODEL_NAME,
        "config": asdict(config),
        "manifest_sha256": manifest_hash,
        "parameter_count": sum(p.numel() for p in model.parameters()),
        "preprocessing": "RGB conversion + model default resize + ImageNet normalization only (no retinal preprocessing)",
        "runtime": {
            "torch": torch.__version__, "timm": timm.__version__,
            "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        },
    }
    (run_dir / "config.json").write_text(json.dumps(record, indent=2), encoding="utf-8")

    wb = None
    if config.wandb_mode != "disabled":
        import wandb
        wb = wandb.init(
            project=os.environ.get("CONVNEXTV2_WANDB_PROJECT", "convnextv2-dr-baseline"),
            name=run_name, dir=str(run_dir), mode=config.wandb_mode, config=record,
        )

    history = []
    best_f1, best_loss = -1.0, float("inf")
    try:
        for epoch in range(config.epochs):
            train_loss = train_one_epoch(model, loaders["train"], optimizer, criterion, device)
            val = evaluate(model, loaders["val"], criterion, device)
            improved = val["f1_macro"] > best_f1 or (val["f1_macro"] == best_f1 and val["loss"] < best_loss)
            if improved:
                best_f1, best_loss = val["f1_macro"], val["loss"]
            state = {"epoch": epoch, "model": model.state_dict(), "optimizer": optimizer.state_dict(),
                     "record": record, "best_f1": best_f1, "best_loss": best_loss}
            _save_checkpoint(run_dir / "last.pt", state)
            if improved:
                _save_checkpoint(run_dir / "best.pt", state)
            row = {"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val["loss"],
                   "val_accuracy": val["accuracy"], "val_f1_macro": val["f1_macro"],
                   "val_qwk": val["qwk"], "best_val_f1_macro": best_f1}
            history.append(row)
            print(row)
            if wb is not None:
                wb.log(row, step=epoch + 1)
    finally:
        if wb is not None:
            wb.finish()

    pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)

    best_state = torch.load(run_dir / "best.pt", map_location=device, weights_only=False)
    model.load_state_dict(best_state["model"])

    for split_name in ("val", "test"):
        result = evaluate(model, loaders[split_name], criterion, device)
        report = classification_report(
            result["y_true"], result["y_pred"],
            target_names=[DR_LABELS[k] for k in range(5)], zero_division=0,
        )
        (run_dir / f"{split_name}_classification_report.txt").write_text(report, encoding="utf-8")
        pd.DataFrame(result["confusion_matrix"], index=list(DR_LABELS.values()),
                     columns=list(DR_LABELS.values())).to_csv(run_dir / f"{split_name}_confusion_matrix.csv")
        pd.DataFrame({"y_true": result["y_true"], "y_pred": result["y_pred"]}).to_csv(
            run_dir / f"{split_name}_predictions.csv", index=False)
        summary = {k: v for k, v in result.items() if k not in ("y_true", "y_pred", "confusion_matrix")}
        (run_dir / f"{split_name}_metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
        print(f"\n=== {split_name.upper()} ({config.dataset}, {MODEL_NAME}) ===")
        print(report)
        print("Macro-F1:", summary["f1_macro"], "| QWK:", summary["qwk"])

    print("\nRun tersimpan di:", run_dir)
    return run_dir


## 7. Menjalankan baseline Stage 1

Dua sel di bawah masing-masing menghasilkan manifest independen dan menjalankan baseline untuk APTOS2019 dan (jika tersedia) DDR. `run_name` memuat `fp32_seed42` agar jelas dibedakan dari run Stage 2-4 nanti.

In [12]:
def train_one_epoch(model, loader, optimizer, criterion, device, log_every=20):
    model.train()
    total_loss, n = 0.0, 0
    for step, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        n += len(y)
        if (step + 1) % log_every == 0 or (step + 1) == len(loader):
            print(f"  batch {step + 1}/{len(loader)} | running train loss {total_loss / n:.4f}")
    return total_loss / n

In [15]:
from pathlib import Path
import shutil

LOCAL_APTOS_ROOT = Path("/content/APTOS2019_local")
if not LOCAL_APTOS_ROOT.is_dir():
    shutil.copytree(APTOS_ROOT, LOCAL_APTOS_ROOT)
APTOS_ROOT = LOCAL_APTOS_ROOT

In [16]:
aptos_manifest = prepare_manifest("APTOS2019")
aptos_config = BaselineConfig(dataset="APTOS2019", wandb_mode="online")
run_baseline(aptos_config, aptos_manifest, run_name="aptos2019_convnextv2tiny_baseline1_fp32_seed42")


Dataset: APTOS2019 | total gambar: 3662 | grup SHA-256: 3534
Jumlah gambar per split: {'train': 2930, 'val': 366, 'test': 366}
Exact-duplicate leakage: 0
Manifest: /content/drive/MyDrive/ConvNeXtV2-DR/manifests/APTOS2019_convnextv2tiny_seed42.csv


model.safetensors: reconstructing file:   0%|          |  0.00B /  115MB            

model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: oraclep8 (oraclep8-brawijaya-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  batch 20/92 | running train loss 0.8476
  batch 40/92 | running train loss 0.7298
  batch 60/92 | running train loss 0.6800
  batch 80/92 | running train loss 0.6431
  batch 92/92 | running train loss 0.6264
{'epoch': 1, 'train_loss': 0.6263595167687321, 'val_loss': 0.5448396430640924, 'val_accuracy': 0.7896174863387978, 'val_f1_macro': 0.4814953379533149, 'val_qwk': 0.8464225582602949, 'best_val_f1_macro': 0.4814953379533149}
  batch 20/92 | running train loss 0.4170
  batch 40/92 | running train loss 0.4187
  batch 60/92 | running train loss 0.4132
  batch 80/92 | running train loss 0.4074
  batch 92/92 | running train loss 0.4073
{'epoch': 2, 'train_loss': 0.4072560442389075, 'val_loss': 0.5218377938198913, 'val_accuracy': 0.8087431693989071, 'val_f1_macro': 0.5990696025245541, 'val_qwk': 0.8910699437015227, 'best_val_f1_macro': 0.5990696025245541}
  batch 20/92 | running train loss 0.2908
  batch 40/92 | running train loss 0.2794
  batch 60/92 | running train loss 0.2815
  batch 

best_val_f1_macro,▁▅██████████████████████████████████████
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_loss,█▅▄▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▄█▄▃▄▄██▅▅▆▅▇▅▅▆▅▄▆▂▂▆▃▅▅▃▅▃▅▅▃▄▅▂▂▅▃▅▃
val_f1_macro,▁▅█▇▅▆▇▇█▇▇▇█▇▇▇▇▇▇▇▇▇▆▇▇▇▇▇▇▇▆█▇▆▇█▆▇▇▇
val_loss,▂▁▁▂▃▂▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▇▅▆▅▆▆▇▆▇▇▇▇▇▇▇█▇██
val_qwk,▁▆▆▆█▆▆▆▆█▇▇▇▇▆▇▆▆▅▇▆▆▅▇▇▆▅▇▆▇▆▅▆▅▅▅▆▅▅▅
best_val_f1_macro,0.66924
epoch,50
train_loss,0.01675
val_accuracy,0.80055



=== VAL (APTOS2019, convnextv2_tiny.fcmae_ft_in22k_in1k) ===
                  precision    recall  f1-score   support

           No DR       0.99      0.98      0.99       180
            Mild       0.53      0.49      0.51        35
        Moderate       0.69      0.81      0.75       101
          Severe       0.44      0.40      0.42        20
Proliferative DR       0.85      0.57      0.68        30

        accuracy                           0.82       366
       macro avg       0.70      0.65      0.67       366
    weighted avg       0.83      0.82      0.82       366

Macro-F1: 0.6692401162554308 | QWK: 0.9050443855441962

=== TEST (APTOS2019, convnextv2_tiny.fcmae_ft_in22k_in1k) ===
                  precision    recall  f1-score   support

           No DR       0.99      0.98      0.99       180
            Mild       0.86      0.66      0.75        38
        Moderate       0.71      0.87      0.78       100
          Severe       0.24      0.26      0.25        19
Prol

PosixPath('/content/drive/MyDrive/ConvNeXtV2-DR/checkpoints/aptos2019_convnextv2tiny_baseline1_fp32_seed42')

In [ ]:
if DDR_ROOT.is_dir():
    ddr_manifest = prepare_manifest("DDR")
    ddr_config = BaselineConfig(dataset="DDR", wandb_mode="offline")
    run_baseline(ddr_config, ddr_manifest, run_name="ddr_convnextv2tiny_baseline_fp32_seed42")
else:
    print("DDR_ROOT belum tersedia; lewati baseline DDR untuk saat ini.")


## 8. Catatan dan langkah selanjutnya

- **Yang EXPLICIT dari brief dan diterapkan apa adanya:** arsitektur ConvNeXtV2-Tiny, bobot ImageNet pretrained, head 5 kelas, input RGB 3 kanal, tanpa preprocessing retina tambahan, augmentasi nonaktif, class balancing nonaktif, CrossEntropyLoss standar, precision FP32.
- **Yang merupakan implementation assumption (bukan dari brief), harus dicatat demikian pada laporan:** jumlah epoch (50), learning rate (5e-5), weight decay (0.05), batch size (32), optimizer AdamW, dan kriteria pemilihan checkpoint (validation macro-F1).
- Notebook ini memenuhi *"Immediate development priority"* pada brief: pipeline baseline ConvNeXtV2-Tiny yang berjalan, manifest independen, dan output eksperimen terpisah dari CAD-ViT -- sebagai dasar sebelum Stage 2 (augmentasi/balancing/severity-aware learning, RQ1), Stage 3 (CPU inference benchmarking, RQ2), dan Stage 4 (perbandingan precision FP32/FP16/INT8, RQ3).
- EDA historis bergaya CAD-ViT (cropping, resize, Gaussian blur, perbandingan Wiener/Ben Graham) **sengaja tidak disertakan** di notebook ini karena baseline ConvNeXtV2-Tiny secara eksplisit tidak memakai preprocessing retina apa pun -- EDA semacam itu relevan lagi saat Stage 2 menguji augmentasi/preprocessing retina-spesifik untuk ConvNeXtV2-Tiny.
- Sebelum melanjutkan ke RQ1 Stage 2, verifikasi dashboard W&B secara manual untuk memastikan metrik run baseline benar-benar ter-log (lihat catatan SS6 di atas).


## 9. Stage 3 — CPU Inference Benchmarking (RQ2)

**Tujuan (brief RQ2):** mengukur apakah latency inference CPU dapat dikurangi melalui optimasi model/runtime, sambil mempertahankan Macro-F1, ordinal agreement (QWK), dan sensitivitas kelas minoritas.

**Aturan dari brief yang diterapkan apa adanya:**
- **Tidak melatih ulang model** -- benchmark memuat `best.pt` dari run baseline Stage 1 FP32 (RQ1) secara langsung.
- Backend yang diuji: **PyTorch CPU (eager)**, **TorchScript** (justified CPU optimization technique -- graph-mode, tanpa precision/arsitektur berubah), **ONNX Runtime**, dan **OpenVINO** (opsional, *"where appropriate"* -- dilewati otomatis dengan pesan jelas bila tidak terpasang).
- Kondisi benchmarking dijaga konsisten lintas backend: **CPU hardware sama** (environment yang menjalankan notebook), **jumlah thread CPU tetap** (`torch.set_num_threads`), **sampel evaluasi identik** (subset test yang sama persis untuk semua backend), **batch size konsisten**, **warm-up iterations**, **pengukuran berulang** (median, bukan rata-rata, agar tahan outlier), dan **waktu preprocessing dipisahkan dari waktu inference model** (`time_preprocessing` vs `benchmark_*_cpu`).
- Diukur: **median latency (ms/image)**, **throughput (images/second)**, **ukuran model** (file checkpoint per backend), dan **computational requirements** (jumlah parameter; FLOPs bila `fvcore` terpasang).
- **Kualitas klasifikasi diperiksa ulang per backend** pada sampel benchmark yang sama dan dibandingkan terhadap metrik test Stage 1 FP32 (Macro-F1, QWK, recall per kelas termasuk kelas minoritas), karena brief secara eksplisit meminta pemeriksaan ini -- optimasi runtime yang lebih cepat tetapi merusak recall kelas minoritas TIDAK dianggap berhasil.

**Implementation assumptions (tidak ditetapkan brief), harus dicatat pada laporan:** `batch_size=1` untuk benchmark (skenario deployment CPU single-image umum), `n_warmup=10`, `n_repeats=50`, `max_eval_samples=200`, toleransi preservasi metrik ±0.02 pada perbandingan kualitas klasifikasi antar backend.

In [42]:
import time
import shutil
import tempfile

from dataclasses import dataclass


@dataclass
class BenchmarkConfig:
    dataset: str = "APTOS2019"
    run_name: str = "aptos2019_convnextv2tiny_baseline_fp32_seed42"  # run Stage 1 yang dibenchmark
    cpu_threads: int = 4            # implementation assumption -- samakan di seluruh backend
    batch_size: int = 1             # implementation assumption -- skenario deployment single-image
    n_warmup: int = 10              # implementation assumption
    n_repeats: int = 5             # implementation assumption
    max_eval_samples: int = 30     # implementation assumption -- identik lintas backend
    metric_tolerance: float = 0.02  # implementation assumption -- ambang "preserved" pada perbandingan kualitas
    quality_eval_samples: int = 150


In [18]:
def prepare_benchmark_samples(manifest_path, dataset, n_samples):
    '''Ambil n_samples contoh TEST pertama (urut image_id, deterministik) -- identik untuk semua backend.'''
    frame = pd.read_csv(manifest_path, dtype={"image_id": "string"})
    test = frame[frame.split == "test"].sort_values("image_id").reset_index(drop=True)
    if len(test) < n_samples:
        print(f"PERINGATAN: test set hanya {len(test)} gambar (< max_eval_samples={n_samples}); memakai semua.")
        n_samples = len(test)
    subset = test.head(n_samples).reset_index(drop=True)
    root = APTOS_ROOT if dataset == "APTOS2019" else DDR_ROOT
    return subset, root


def time_preprocessing(frame, root, n_repeats=20):
    '''Waktu transform (RGB convert + resize + ImageNet normalize) TERPISAH dari model inference.'''
    paths = [root / p for p in frame["image_path"]]
    for p in paths[: min(5, len(paths))]:
        with Image.open(p) as img:
            BASELINE_TRANSFORM(ImageOps.exif_transpose(img).convert("RGB"))
    per_image_ms = []
    for _ in range(n_repeats):
        start = time.perf_counter()
        for p in paths:
            with Image.open(p) as img:
                BASELINE_TRANSFORM(ImageOps.exif_transpose(img).convert("RGB"))
        per_image_ms.append((time.perf_counter() - start) / len(paths) * 1000)
    return {
        "median_ms_per_image": float(np.median(per_image_ms)),
        "n_images": len(paths),
        "n_repeats": n_repeats,
    }


def _build_fixed_batches(frame, root, batch_size):
    '''Tensor input tetap (sudah melalui transform baseline), identik untuk semua backend.'''
    tensors, labels = [], []
    for row in frame.itertuples(index=False):
        with Image.open(root / row.image_path) as img:
            tensors.append(BASELINE_TRANSFORM(ImageOps.exif_transpose(img).convert("RGB")))
        labels.append(int(row.label))
    x = torch.stack(tensors)
    y = torch.tensor(labels, dtype=torch.long)
    n_batches = len(x) // batch_size
    if n_batches == 0:
        raise ValueError("max_eval_samples harus >= batch_size")
    x = x[: n_batches * batch_size].reshape(n_batches, batch_size, *x.shape[1:])
    y = y[: n_batches * batch_size].reshape(n_batches, batch_size)
    return x, y


def file_size_mb(path):
    return Path(path).stat().st_size / (1024 * 1024)

### 9.1 Backend: PyTorch CPU (eager) dan TorchScript

In [19]:
@torch.no_grad()
def benchmark_pytorch_cpu(model, batches, n_warmup, n_repeats):
    model.eval()
    model.to("cpu")
    for i in range(min(n_warmup, len(batches))):
        model(batches[i % len(batches)])
    latencies = []
    for _ in range(n_repeats):
        for batch in batches:
            start = time.perf_counter()
            model(batch)
            latencies.append((time.perf_counter() - start) / batch.shape[0] * 1000)
    return {
        "backend": "pytorch_cpu_eager",
        "median_latency_ms_per_image": float(np.median(latencies)),
        "throughput_images_per_second": float(1000.0 / np.median(latencies)),
        "n_measurements": len(latencies),
    }


@torch.no_grad()
def classification_quality_pytorch(model, x_batches, y_batches):
    model.eval()
    true, pred = [], []
    for x, y in zip(x_batches, y_batches):
        logits = model(x)
        true.extend(y.tolist())
        pred.extend(logits.argmax(1).tolist())
    return compute_metrics(true, pred)


def export_torchscript(model, path):
    model.eval()
    dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
    traced = torch.jit.trace(model, dummy)
    traced.save(str(path))
    return path


@torch.no_grad()
def benchmark_torchscript_cpu(ts_path, batches, n_warmup, n_repeats):
    model = torch.jit.load(str(ts_path))
    model.eval()
    for i in range(min(n_warmup, len(batches))):
        model(batches[i % len(batches)])
    latencies = []
    for _ in range(n_repeats):
        for batch in batches:
            start = time.perf_counter()
            model(batch)
            latencies.append((time.perf_counter() - start) / batch.shape[0] * 1000)
    return {
        "backend": "torchscript_cpu",
        "median_latency_ms_per_image": float(np.median(latencies)),
        "throughput_images_per_second": float(1000.0 / np.median(latencies)),
        "n_measurements": len(latencies),
    }


@torch.no_grad()
def classification_quality_torchscript(ts_path, x_batches, y_batches):
    model = torch.jit.load(str(ts_path))
    model.eval()
    true, pred = [], []
    for x, y in zip(x_batches, y_batches):
        logits = model(x)
        true.extend(y.tolist())
        pred.extend(logits.argmax(1).tolist())
    return compute_metrics(true, pred)

### 9.2 Backend: ONNX Runtime

In [20]:
def export_onnx(model, path):
    '''Ekspor ke ONNX via exporter legacy (TorchScript-based, dynamo=False).

    torch>=2.5 secara default memakai exporter 'dynamo' yang membutuhkan paket tambahan
    `onnxscript` dan belum tentu terpasang di semua runtime Colab/CPU. Exporter legacy
    jauh lebih stabil untuk CNN seperti ConvNeXtV2 dan tidak butuh dependency tambahan.'''
    try:
        import onnx  # noqa: F401  -- dibutuhkan exporter legacy untuk serialisasi proto
    except ImportError:
        import sys
        get_ipython().system(f"{sys.executable} -m pip install -q onnx")
    model.eval()
    dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE)
    export_kwargs = dict(
        input_names=["input"], output_names=["logits"],
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17,
    )
    try:
        torch.onnx.export(model, dummy, str(path), dynamo=False, **export_kwargs)
    except TypeError:
        # torch versions without the `dynamo` kwarg always use the legacy exporter.
        torch.onnx.export(model, dummy, str(path), **export_kwargs)
    return path


def _get_onnxruntime(cpu_threads):
    try:
        import onnxruntime as ort
    except ImportError:
        import sys
        get_ipython().system(f"{sys.executable} -m pip install -q onnxruntime")
        import onnxruntime as ort
    so = ort.SessionOptions()
    so.intra_op_num_threads = cpu_threads
    so.inter_op_num_threads = 1
    return ort, so


def benchmark_onnxruntime_cpu(onnx_path, batches, n_warmup, n_repeats, cpu_threads):
    ort, so = _get_onnxruntime(cpu_threads)
    session = ort.InferenceSession(str(onnx_path), sess_options=so, providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name

    def run(batch):
        return session.run(None, {input_name: batch.numpy()})[0]

    for i in range(min(n_warmup, len(batches))):
        run(batches[i % len(batches)])
    latencies = []
    for _ in range(n_repeats):
        for batch in batches:
            start = time.perf_counter()
            run(batch)
            latencies.append((time.perf_counter() - start) / batch.shape[0] * 1000)
    return {
        "backend": "onnxruntime_cpu",
        "median_latency_ms_per_image": float(np.median(latencies)),
        "throughput_images_per_second": float(1000.0 / np.median(latencies)),
        "n_measurements": len(latencies),
    }


def classification_quality_onnx(onnx_path, x_batches, y_batches, cpu_threads):
    ort, so = _get_onnxruntime(cpu_threads)
    session = ort.InferenceSession(str(onnx_path), sess_options=so, providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name
    true, pred = [], []
    for x, y in zip(x_batches, y_batches):
        logits = session.run(None, {input_name: x.numpy()})[0]
        true.extend(y.tolist())
        pred.extend(np.argmax(logits, axis=1).tolist())
    return compute_metrics(true, pred)


### 9.3 Backend: OpenVINO (opsional -- *"where appropriate"*)

In [21]:
def benchmark_openvino_cpu(onnx_path, batches, n_warmup, n_repeats, cpu_threads):
    '''Dilewati otomatis (return None) bila OpenVINO tidak terpasang -- brief: 'where appropriate'.'''
    try:
        import openvino as ov
    except ImportError:
        print("OpenVINO tidak terpasang; lewati backend ini (opsional menurut brief RQ2).")
        return None, None
    core = ov.Core()
    ov_model = core.read_model(str(onnx_path))
    compiled = core.compile_model(ov_model, "CPU", config={"INFERENCE_NUM_THREADS": str(cpu_threads)})
    output_port = compiled.output(0)

    def run(batch):
        return compiled([batch.numpy()])[output_port]

    for i in range(min(n_warmup, len(batches))):
        run(batches[i % len(batches)])
    latencies = []
    for _ in range(n_repeats):
        for batch in batches:
            start = time.perf_counter()
            run(batch)
            latencies.append((time.perf_counter() - start) / batch.shape[0] * 1000)
    timing = {
        "backend": "openvino_cpu",
        "median_latency_ms_per_image": float(np.median(latencies)),
        "throughput_images_per_second": float(1000.0 / np.median(latencies)),
        "n_measurements": len(latencies),
    }
    return timing, compiled


def classification_quality_openvino(compiled, x_batches, y_batches):
    if compiled is None:
        return None
    output_port = compiled.output(0)
    true, pred = [], []
    for x, y in zip(x_batches, y_batches):
        logits = compiled([x.numpy()])[output_port]
        true.extend(y.tolist())
        pred.extend(np.argmax(logits, axis=1).tolist())
    return compute_metrics(true, pred)

### 9.4 Computational requirements (jumlah parameter dan FLOPs opsional)

In [22]:
def computational_requirements(model):
    result = {"parameter_count": sum(p.numel() for p in model.parameters())}
    try:
        from fvcore.nn import FlopCountAnalysis
        flops = FlopCountAnalysis(model, torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE))
        flops.unsupported_ops_warnings(False)
        result["flops_per_image"] = int(flops.total())
    except ImportError:
        result["flops_per_image"] = None
        print("fvcore tidak terpasang -- FLOPs tidak dihitung (pip install fvcore untuk mengaktifkan).")
    return result


### 9.5 Orkestrasi: jalankan seluruh benchmark dan periksa preservasi kualitas

`run_cpu_benchmark` memuat `best.pt` dari run Stage 1 FP32 yang diberikan (**tidak melatih ulang**), menjalankan keempat backend pada sampel test yang identik, dan membandingkan Macro-F1/QWK/recall per kelas tiap backend terhadap metrik test Stage 1 FP32 asli (`test_metrics.json`) untuk memverifikasi bahwa optimasi CPU tidak diam-diam merusak kualitas klasifikasi -- terutama kelas minoritas (Severe, Proliferative DR).

In [43]:
def compare_against_stage1(stage1_test_metrics_path, backend_metrics, tolerance):
    baseline = json.loads(Path(stage1_test_metrics_path).read_text(encoding="utf-8"))
    rows = []
    for key in ("f1_macro", "qwk"):
        delta = backend_metrics[key] - baseline[key]
        rows.append({"metric": key, "stage1_fp32_test": baseline[key], "backend_benchmark_subset": backend_metrics[key],
                     "delta": delta, "preserved": abs(delta) <= tolerance})
    for cls, recall in backend_metrics["per_class_recall"].items():
        base_recall = baseline["per_class_recall"][cls]
        delta = recall - base_recall
        rows.append({"metric": f"recall[{cls}]", "stage1_fp32_test": base_recall, "backend_benchmark_subset": recall,
                     "delta": delta, "preserved": abs(delta) <= tolerance})
    return pd.DataFrame(rows)


def run_cpu_benchmark(config: BenchmarkConfig):
    run_dir = RUN_ROOT / "checkpoints" / config.run_name
    best_ckpt = run_dir / "best.pt"
    stage1_test_metrics = run_dir / "test_metrics.json"
    if not best_ckpt.is_file():
        raise FileNotFoundError(f"Checkpoint Stage 1 tidak ditemukan: {best_ckpt}")
    if not stage1_test_metrics.is_file():
        raise FileNotFoundError(f"test_metrics.json Stage 1 tidak ditemukan: {stage1_test_metrics}")

    torch.set_num_threads(config.cpu_threads)
    device = torch.device("cpu")

    state = torch.load(best_ckpt, map_location=device, weights_only=False)
    if state["record"]["config"]["precision"] != "fp32":
        raise ValueError("RQ2 wajib memakai checkpoint FP32 Stage 1 (brief: 'Use the best FP32 checkpoint').")
    model = build_model(num_classes=5, pretrained=False)
    model.load_state_dict(state["model"])
    model.eval()

    manifest_path = RUN_ROOT / "manifests" / f"{config.dataset}_convnextv2tiny_seed{SPLIT_SEED}.csv"

    # --- Sampel untuk TIMING (kecil, diulang n_repeats x -- menentukan lama benchmark) ---
    frame, root = prepare_benchmark_samples(manifest_path, config.dataset, config.max_eval_samples)
    preprocessing_timing = time_preprocessing(frame, root)
    x_batches, y_batches = _build_fixed_batches(frame, root, config.batch_size)

    # --- Sampel TERPISAH untuk KUALITAS (lebih besar, hanya SATU pass per backend --
    #     tidak dikali n_repeats, sehingga tidak memperlambat benchmark secara proporsional).
    quality_frame, _ = prepare_benchmark_samples(manifest_path, config.dataset, config.quality_eval_samples)
    quality_x_batches, quality_y_batches = _build_fixed_batches(quality_frame, root, config.batch_size)

    bench_dir = RUN_ROOT / "benchmarks" / config.run_name
    bench_dir.mkdir(parents=True, exist_ok=True)

    results = {
        "run_name": config.run_name, "dataset": config.dataset,
        "config": asdict(config), "preprocessing_timing": preprocessing_timing,
        "n_quality_eval_samples_actual": len(quality_frame),
        "computational_requirements": computational_requirements(model),
        "cpu_info": {"torch_num_threads": torch.get_num_threads()},
        "backends": {},
        "quality_vs_stage1_fp32": {},
    }

    # --- PyTorch eager ---
    timing = benchmark_pytorch_cpu(model, list(x_batches), config.n_warmup, config.n_repeats)
    quality = classification_quality_pytorch(model, quality_x_batches, quality_y_batches)
    state_only_path = bench_dir / "model_state_fp32.pt"
    torch.save(model.state_dict(), state_only_path)
    timing["model_size_mb"] = file_size_mb(state_only_path)
    results["backends"]["pytorch_cpu_eager"] = {**timing, "quality": {k: v for k, v in quality.items() if k != "confusion_matrix"}}
    results["quality_vs_stage1_fp32"]["pytorch_cpu_eager"] = compare_against_stage1(
        stage1_test_metrics, quality, config.metric_tolerance).to_dict(orient="records")

    # --- TorchScript ---
    ts_path = bench_dir / "model_fp32.torchscript.pt"
    export_torchscript(model, ts_path)
    timing = benchmark_torchscript_cpu(ts_path, list(x_batches), config.n_warmup, config.n_repeats)
    quality = classification_quality_torchscript(ts_path, quality_x_batches, quality_y_batches)
    timing["model_size_mb"] = file_size_mb(ts_path)
    results["backends"]["torchscript_cpu"] = {**timing, "quality": {k: v for k, v in quality.items() if k != "confusion_matrix"}}
    results["quality_vs_stage1_fp32"]["torchscript_cpu"] = compare_against_stage1(
        stage1_test_metrics, quality, config.metric_tolerance).to_dict(orient="records")

    # --- ONNX Runtime ---
    onnx_path = bench_dir / "model_fp32.onnx"
    export_onnx(model, onnx_path)
    timing = benchmark_onnxruntime_cpu(onnx_path, list(x_batches), config.n_warmup, config.n_repeats, config.cpu_threads)
    quality = classification_quality_onnx(onnx_path, quality_x_batches, quality_y_batches, config.cpu_threads)
    timing["model_size_mb"] = file_size_mb(onnx_path)
    results["backends"]["onnxruntime_cpu"] = {**timing, "quality": {k: v for k, v in quality.items() if k != "confusion_matrix"}}
    results["quality_vs_stage1_fp32"]["onnxruntime_cpu"] = compare_against_stage1(
        stage1_test_metrics, quality, config.metric_tolerance).to_dict(orient="records")

    # --- OpenVINO (optional) ---
    ov_timing, ov_compiled = benchmark_openvino_cpu(onnx_path, list(x_batches), config.n_warmup, config.n_repeats, config.cpu_threads)
    if ov_timing is not None:
        ov_quality = classification_quality_openvino(ov_compiled, quality_x_batches, quality_y_batches)
        ov_timing["model_size_mb"] = file_size_mb(onnx_path)
        results["backends"]["openvino_cpu"] = {**ov_timing, "quality": {k: v for k, v in ov_quality.items() if k != "confusion_matrix"}}
        results["quality_vs_stage1_fp32"]["openvino_cpu"] = compare_against_stage1(
            stage1_test_metrics, ov_quality, config.metric_tolerance).to_dict(orient="records")

    out_path = bench_dir / "cpu_benchmark.json"
    out_path.write_text(json.dumps(results, indent=2), encoding="utf-8")

    summary_rows = [
        {"backend": name, "median_latency_ms_per_image": r["median_latency_ms_per_image"],
         "throughput_images_per_second": r["throughput_images_per_second"],
         "model_size_mb": r["model_size_mb"], "f1_macro": r["quality"]["f1_macro"], "qwk": r["quality"]["qwk"]}
        for name, r in results["backends"].items()
    ]
    summary_df = pd.DataFrame(summary_rows).sort_values("median_latency_ms_per_image")
    print(f"\n=== Ringkasan CPU Inference Benchmark ({config.dataset}, {config.run_name}) ===")
    print(f"preprocessing: {preprocessing_timing['median_ms_per_image']:.3f} ms/image (di luar model, sama untuk semua backend)")
    print(f"timing dihitung dari {config.max_eval_samples} sampel x {config.n_repeats} repeats; "
          f"kualitas dihitung dari {len(quality_frame)} sampel terpisah (satu pass, tidak dikali n_repeats)")
    print(summary_df.to_string(index=False))

    for name, rows in results["quality_vs_stage1_fp32"].items():
        not_preserved = [row for row in rows if not row["preserved"]]
        if not_preserved:
            print(f"\nPERINGATAN -- {name}: metrik berikut BERUBAH melampaui toleransi ±{config.metric_tolerance} "
                  f"dibanding Stage 1 FP32 test:")
            for row in not_preserved:
                print(f"  {row['metric']}: {row['stage1_fp32_test']:.4f} -> {row['backend_benchmark_subset']:.4f} "
                      f"(delta {row['delta']:+.4f})")

    print("\nHasil lengkap tersimpan di:", out_path)
    return results

### 9.6 Menjalankan benchmark

Jalankan setelah run Stage 1 FP32 untuk dataset terkait sudah selesai (cell §7). `run_name` harus cocok persis dengan nama run Stage 1 yang ingin dibenchmark.

In [45]:
aptos_benchmark_config = BenchmarkConfig(
    dataset="APTOS2019",
    run_name="aptos2019_convnextv2tiny_baseline1_fp32_seed42",
    max_eval_samples=30,
    n_repeats=20,
    quality_eval_samples=150,
)
aptos_benchmark_results = run_cpu_benchmark(aptos_benchmark_config)

/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/tmp/ipykernel_1440/44310885.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, str(pat

OpenVINO tidak terpasang; lewati backend ini (opsional menurut brief RQ2).

=== Ringkasan CPU Inference Benchmark (APTOS2019, aptos2019_convnextv2tiny_baseline1_fp32_seed42) ===
preprocessing: 147.577 ms/image (di luar model, sama untuk semua backend)
timing dihitung dari 30 sampel x 20 repeats; kualitas dihitung dari 150 sampel terpisah (satu pass, tidak dikali n_repeats)
          backend  median_latency_ms_per_image  throughput_images_per_second  model_size_mb  f1_macro      qwk
  onnxruntime_cpu                    42.609615                     23.468881     106.427517  0.610661 0.895682
  torchscript_cpu                    55.887753                     17.893008     106.792458  0.610661 0.895682
pytorch_cpu_eager                    63.144897                     15.836592     106.388593  0.610661 0.895682

PERINGATAN -- pytorch_cpu_eager: metrik berikut BERUBAH melampaui toleransi ±0.02 dibanding Stage 1 FP32 test:
  f1_macro: 0.6402 -> 0.6107 (delta -0.0295)
  recall[No DR]: 0.9778

In [ ]:
if DDR_ROOT.is_dir():
    ddr_benchmark_config = BenchmarkConfig(
        dataset="DDR",
        run_name="ddr_convnextv2tiny_baseline_fp32_seed42",
    )
    ddr_benchmark_results = run_cpu_benchmark(ddr_benchmark_config)
else:
    print("DDR_ROOT belum tersedia; lewati benchmark DDR untuk saat ini.")


## 10. Structured Pruning — L1-norm, ratio sweep (eksplorasi tambahan di luar RQ1-RQ3)

**Posisi terhadap brief:** bagian ini **bukan** bagian dari RQ1/RQ2/RQ3 yang ditetapkan brief -- brief hanya meminta "investigate model and runtime optimization" pada RQ2 tanpa menetapkan teknik spesifik. Structured pruning di sini adalah eksplorasi tambahan atas inisiatif sendiri untuk memperdalam RQ2, dan harus ditandai secara eksplisit sebagai eksplorasi tambahan (bukan requirement brief) pada laporan.

**Scope -- disengaja dibatasi agar tidak butuh dependency-graph tooling:**
- Yang dipangkas: **dimensi ekspansi 4x** di dalam tiap ConvNeXtV2 block (`mlp.fc1` -> `mlp.grn` -> `mlp.fc2`), BUKAN dimensi backbone (`dim`) yang mengalir lewat residual connection antar block.
- Karena `mlp.fc1` menerima `dim` (tidak berubah) dan `mlp.fc2` mengeluarkan `dim` (tidak berubah) -- yang berubah cuma lebar tersembunyi `4*dim` di antara keduanya -- pemangkasan ini **block-local**: tidak ada propagasi dependency ke block lain, residual (`shortcut`) atau layer-scale (`gamma`), sehingga tidak memerlukan tool dependency-graph seperti `torch-pruning`.
- Kriteria importance: **L1-norm** dari tiap baris `fc1.weight` (baris = satu channel ekspansi) -- channel dengan L1-norm terkecil dibuang (kriteria Li et al., diadaptasi dari Conv filter ke Linear "filter").
- Sweep rasio pruning: **10% / 30% / 50%** dari lebar ekspansi (4*dim channel), per block, seragam di semua block.
- Setiap rasio di-**fine-tune singkat** (recovery, BUKAN training dari awal -- bobot awal fine-tuning adalah checkpoint FP32 Stage 1 yang sudah di-prune) untuk memulihkan sebagian akurasi yang hilang akibat pemangkasan.
- Setiap rasio di-benchmark CPU memakai **fungsi yang sama persis dari §9** (`benchmark_pytorch_cpu`, `benchmark_torchscript_cpu`, `benchmark_onnxruntime_cpu`, `benchmark_openvino_cpu`, `classification_quality_*`, `export_torchscript`, `export_onnx`, `compare_against_stage1`) -- orkestrasinya (`run_cpu_benchmark_pruned`) dibuat terpisah dari `run_cpu_benchmark` karena arsitektur hasil pruning tidak lagi cocok dimuat lewat `build_model()` + `load_state_dict()` baku (ukuran `fc1`/`grn`/`fc2` sudah berbeda), bukan karena logika benchmark-nya ditulis ulang.
- Perbandingan kualitas tetap terhadap **test metrics Stage 1 FP32 asli** (`test_metrics.json` di `base_run_name`), bukan terhadap model pruned sebelum fine-tuning -- supaya pertanyaan yang dijawab konsisten dengan RQ2: apakah hasil akhir (setelah optimasi) masih comparable dengan baseline asli.

**Implementation assumptions (harus dicatat pada laporan):** `prune_ratios=(0.10, 0.30, 0.50)`, `finetune_epochs=3`, `finetune_lr=1e-5` (lebih kecil dari `learning_rate` baseline 5e-5, karena ini recovery bukan training awal), `finetune_batch_size=32`, kriteria importance L1-norm per block (bukan global lintas block), fine-tuning dijalankan di GPU bila tersedia (lebih cepat), evaluasi akhir tetap di CPU lewat harness §9.

In [46]:
import copy
from timm.layers.grn import GlobalResponseNorm

def _l1_importance(weight):
    '''L1-norm tiap baris (= satu channel ekspansi) dari weight matrix Linear fc1.'''
    return weight.detach().abs().sum(dim=1)


def _prune_expansion_mlp(mlp, keep_ratio, importance="l1"):
    '''Pangkas dimensi ekspansi (4x) di dalam satu GlobalResponseNormMlp (fc1 -> grn -> fc2) secara
    in-place. keep_ratio = fraksi channel ekspansi yang DIPERTAHANKAN. Hanya mengubah lebar
    tersembunyi internal block ini -- input fc1 (dim) dan output fc2 (dim) tidak berubah, sehingga
    tidak ada dependency ke block/residual lain.'''
    if importance != "l1":
        raise NotImplementedError(f"importance={importance!r} belum diimplementasikan")

    hidden_before = mlp.fc1.out_features
    keep_n = max(1, round(hidden_before * keep_ratio))

    scores = _l1_importance(mlp.fc1.weight)
    keep_idx = torch.argsort(scores, descending=True)[:keep_n].sort().values

    channels_last = isinstance(mlp.fc1, nn.Linear)  # convnextv2_tiny.fcmae_ft_in22k_in1k pakai Linear (bukan Conv2d 1x1)

    new_fc1 = nn.Linear(mlp.fc1.in_features, keep_n, bias=mlp.fc1.bias is not None)
    new_fc1.weight.data = mlp.fc1.weight.data[keep_idx].clone()
    if mlp.fc1.bias is not None:
        new_fc1.bias.data = mlp.fc1.bias.data[keep_idx].clone()

    new_grn = GlobalResponseNorm(keep_n, eps=mlp.grn.eps, channels_last=channels_last)
    new_grn.weight.data = mlp.grn.weight.data[keep_idx].clone()
    new_grn.bias.data = mlp.grn.bias.data[keep_idx].clone()

    new_fc2 = nn.Linear(keep_n, mlp.fc2.out_features, bias=mlp.fc2.bias is not None)
    new_fc2.weight.data = mlp.fc2.weight.data[:, keep_idx].clone()
    if mlp.fc2.bias is not None:
        new_fc2.bias.data = mlp.fc2.bias.data.clone()  # dimensi output (dim) tidak berubah

    mlp.fc1 = new_fc1
    mlp.grn = new_grn
    mlp.fc2 = new_fc2
    return hidden_before, keep_n


def prune_model_expansion(model, keep_ratio, importance="l1"):
    '''Terapkan structured pruning L1-norm pada dimensi ekspansi tiap ConvNeXtV2 block, in-place,
    di seluruh model.stages. keep_ratio = fraksi channel ekspansi yang DIPERTAHANKAN (bukan dibuang).'''
    summary = []
    for stage_idx, stage in enumerate(model.stages):
        for block_idx, block in enumerate(stage.blocks):
            hidden_before, hidden_after = _prune_expansion_mlp(block.mlp, keep_ratio, importance)
            summary.append({
                "stage": stage_idx, "block": block_idx,
                "hidden_before": hidden_before, "hidden_after": hidden_after,
            })
    return pd.DataFrame(summary)


In [47]:
@dataclass
class PruningConfig:
    base_run_name: str                        # run Stage 1 FP32 yang akan di-prune (harus punya best.pt + test_metrics.json)
    dataset: str = "APTOS2019"
    prune_ratios: tuple = (0.10, 0.30, 0.50)   # implementation assumption -- fraksi channel ekspansi yang DIBUANG
    finetune_epochs: int = 3                   # implementation assumption -- recovery, BUKAN training dari awal
    finetune_lr: float = 1e-5                  # implementation assumption -- lebih kecil dari baseline (5e-5)
    finetune_weight_decay: float = 0.05
    finetune_batch_size: int = 32
    finetune_num_workers: int = 2
    seed: int = SPLIT_SEED
    cpu_threads: int = 4
    max_eval_samples: int = 30                 # sampel TIMING CPU benchmark -- lihat §9 untuk rasionalnya
    n_warmup: int = 10
    n_repeats: int = 20
    quality_eval_samples: int = 150            # sampel KUALITAS CPU benchmark, terpisah dari timing (§9)
    metric_tolerance: float = 0.02


### 10.1 Benchmark CPU untuk model pruned (reuse fungsi §9, orkestrasi baru)

Fungsi ini memanggil ulang `benchmark_pytorch_cpu`, `benchmark_torchscript_cpu`, `benchmark_onnxruntime_cpu`, `benchmark_openvino_cpu`, `classification_quality_*`, `export_torchscript`, `export_onnx`, `compare_against_stage1`, `prepare_benchmark_samples`, `time_preprocessing`, `_build_fixed_batches`, `file_size_mb`, `computational_requirements` yang sama persis dari §9 -- bedanya hanya cara model dimuat (objek langsung, bukan lewat `build_model()` + `load_state_dict()`), karena arsitektur pruned punya ukuran `fc1`/`grn`/`fc2` yang berbeda dari model default.

In [48]:
def run_cpu_benchmark_pruned(model, config, stage1_test_metrics_path, bench_dir):
    torch.set_num_threads(config.cpu_threads)
    model = model.to("cpu").eval()

    manifest_path = RUN_ROOT / "manifests" / f"{config.dataset}_convnextv2tiny_seed{SPLIT_SEED}.csv"
    frame, root = prepare_benchmark_samples(manifest_path, config.dataset, config.max_eval_samples)
    preprocessing_timing = time_preprocessing(frame, root)
    x_batches, y_batches = _build_fixed_batches(frame, root, 1)

    quality_frame, _ = prepare_benchmark_samples(manifest_path, config.dataset, config.quality_eval_samples)
    quality_x_batches, quality_y_batches = _build_fixed_batches(quality_frame, root, 1)

    bench_dir.mkdir(parents=True, exist_ok=True)
    results = {
        "dataset": config.dataset, "preprocessing_timing": preprocessing_timing,
        "n_quality_eval_samples_actual": len(quality_frame),
        "computational_requirements": computational_requirements(model),
        "cpu_info": {"torch_num_threads": torch.get_num_threads()},
        "backends": {}, "quality_vs_stage1_fp32": {},
    }

    # --- PyTorch eager ---
    timing = benchmark_pytorch_cpu(model, list(x_batches), config.n_warmup, config.n_repeats)
    quality = classification_quality_pytorch(model, quality_x_batches, quality_y_batches)
    state_only_path = bench_dir / "model_state_fp32.pt"
    torch.save(model.state_dict(), state_only_path)
    timing["model_size_mb"] = file_size_mb(state_only_path)
    results["backends"]["pytorch_cpu_eager"] = {**timing, "quality": {k: v for k, v in quality.items() if k != "confusion_matrix"}}
    results["quality_vs_stage1_fp32"]["pytorch_cpu_eager"] = compare_against_stage1(
        stage1_test_metrics_path, quality, config.metric_tolerance).to_dict(orient="records")

    # --- TorchScript ---
    ts_path = bench_dir / "model_fp32.torchscript.pt"
    export_torchscript(model, ts_path)
    timing = benchmark_torchscript_cpu(ts_path, list(x_batches), config.n_warmup, config.n_repeats)
    quality = classification_quality_torchscript(ts_path, quality_x_batches, quality_y_batches)
    timing["model_size_mb"] = file_size_mb(ts_path)
    results["backends"]["torchscript_cpu"] = {**timing, "quality": {k: v for k, v in quality.items() if k != "confusion_matrix"}}
    results["quality_vs_stage1_fp32"]["torchscript_cpu"] = compare_against_stage1(
        stage1_test_metrics_path, quality, config.metric_tolerance).to_dict(orient="records")

    # --- ONNX Runtime ---
    onnx_path = bench_dir / "model_fp32.onnx"
    export_onnx(model, onnx_path)
    timing = benchmark_onnxruntime_cpu(onnx_path, list(x_batches), config.n_warmup, config.n_repeats, config.cpu_threads)
    quality = classification_quality_onnx(onnx_path, quality_x_batches, quality_y_batches, config.cpu_threads)
    timing["model_size_mb"] = file_size_mb(onnx_path)
    results["backends"]["onnxruntime_cpu"] = {**timing, "quality": {k: v for k, v in quality.items() if k != "confusion_matrix"}}
    results["quality_vs_stage1_fp32"]["onnxruntime_cpu"] = compare_against_stage1(
        stage1_test_metrics_path, quality, config.metric_tolerance).to_dict(orient="records")

    # --- OpenVINO (optional) ---
    ov_timing, ov_compiled = benchmark_openvino_cpu(onnx_path, list(x_batches), config.n_warmup, config.n_repeats, config.cpu_threads)
    if ov_timing is not None:
        ov_quality = classification_quality_openvino(ov_compiled, quality_x_batches, quality_y_batches)
        ov_timing["model_size_mb"] = file_size_mb(onnx_path)
        results["backends"]["openvino_cpu"] = {**ov_timing, "quality": {k: v for k, v in ov_quality.items() if k != "confusion_matrix"}}
        results["quality_vs_stage1_fp32"]["openvino_cpu"] = compare_against_stage1(
            stage1_test_metrics_path, ov_quality, config.metric_tolerance).to_dict(orient="records")

    out_path = bench_dir / "cpu_benchmark.json"
    out_path.write_text(json.dumps(results, indent=2), encoding="utf-8")

    summary_rows = [
        {"backend": name, "median_latency_ms_per_image": r["median_latency_ms_per_image"],
         "throughput_images_per_second": r["throughput_images_per_second"],
         "model_size_mb": r["model_size_mb"], "f1_macro": r["quality"]["f1_macro"], "qwk": r["quality"]["qwk"]}
        for name, r in results["backends"].items()
    ]
    summary_df = pd.DataFrame(summary_rows).sort_values("median_latency_ms_per_image")
    print(f"  timing: {config.max_eval_samples} sampel x {config.n_repeats} repeats; "
          f"kualitas: {len(quality_frame)} sampel terpisah (satu pass)")
    print(summary_df.to_string(index=False))
    print("  Tersimpan di:", out_path)
    return results


### 10.2 Orkestrasi sweep: prune -> fine-tune (recovery) -> benchmark, per rasio

In [49]:
def run_pruning_sweep(config: PruningConfig):
    base_run_dir = RUN_ROOT / "checkpoints" / config.base_run_name
    base_ckpt = base_run_dir / "best.pt"
    stage1_test_metrics = base_run_dir / "test_metrics.json"
    if not base_ckpt.is_file():
        raise FileNotFoundError(f"Checkpoint dasar tidak ditemukan: {base_ckpt}")
    if not stage1_test_metrics.is_file():
        raise FileNotFoundError(f"test_metrics.json dasar tidak ditemukan: {stage1_test_metrics}")

    manifest_path = RUN_ROOT / "manifests" / f"{config.dataset}_convnextv2tiny_seed{SPLIT_SEED}.csv"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Fine-tuning recovery akan berjalan di: {device}")

    loaders = make_loaders_baseline(manifest_path, config.dataset, config.finetune_batch_size,
                                     config.seed, config.finetune_num_workers)

    all_results = {}
    for prune_ratio in config.prune_ratios:
        set_experiment_seed(config.seed)
        print(f"\n########## Pruning ratio: {prune_ratio:.0%} (keep {1 - prune_ratio:.0%} channel ekspansi) ##########")

        state = torch.load(base_ckpt, map_location="cpu", weights_only=False)
        model = build_model(num_classes=5, pretrained=False)
        model.load_state_dict(state["model"])
        model.to(device)

        params_before = sum(p.numel() for p in model.parameters())
        prune_summary = prune_model_expansion(model, keep_ratio=1 - prune_ratio)
        params_after = sum(p.numel() for p in model.parameters())
        print(f"Parameter: {params_before:,} -> {params_after:,} "
              f"({(1 - params_after / params_before):.2%} reduction)")

        pruned_run_name = f"{config.base_run_name}_pruned_l1_{int(round(prune_ratio * 100))}pct"
        pruned_run_dir = RUN_ROOT / "checkpoints" / pruned_run_name
        pruned_run_dir.mkdir(parents=True, exist_ok=True)

        criterion = nn.CrossEntropyLoss()

        # --- Evaluasi SEBELUM fine-tuning (dampak langsung pruning) ---
        pre_ft_metrics = evaluate(model, loaders["test"], criterion, device)
        print(f"Sebelum fine-tuning -- test Macro-F1: {pre_ft_metrics['f1_macro']:.4f}  QWK: {pre_ft_metrics['qwk']:.4f}")

        # --- Fine-tuning recovery singkat (BUKAN training dari awal) ---
        optimizer = torch.optim.AdamW(model.parameters(), lr=config.finetune_lr,
                                       weight_decay=config.finetune_weight_decay)
        best_val_f1, best_state = -1.0, copy.deepcopy(model.state_dict())
        for epoch in range(1, config.finetune_epochs + 1):
            train_loss = train_one_epoch(model, loaders["train"], optimizer, criterion, device)
            val_metrics = evaluate(model, loaders["val"], criterion, device)
            print(f"  [finetune] epoch {epoch}: train_loss={train_loss:.4f} "
                  f"val_f1_macro={val_metrics['f1_macro']:.4f} val_qwk={val_metrics['qwk']:.4f}")
            if val_metrics["f1_macro"] > best_val_f1:
                best_val_f1 = val_metrics["f1_macro"]
                best_state = copy.deepcopy(model.state_dict())
        model.load_state_dict(best_state)

        post_ft_metrics = evaluate(model, loaders["test"], criterion, device)
        print(f"Setelah fine-tuning -- test Macro-F1: {post_ft_metrics['f1_macro']:.4f}  QWK: {post_ft_metrics['qwk']:.4f}")

        # --- Simpan hasil pruning+fine-tuning (model PENUH, bukan hanya state_dict -- arsitektur
        #     fc1/grn/fc2 sudah berubah ukuran sehingga build_model() default tidak cocok lagi) ---
        model_cpu = copy.deepcopy(model).to("cpu").eval()
        torch.save(model_cpu, pruned_run_dir / "pruned_model_full.pt")
        (pruned_run_dir / "test_metrics.json").write_text(
            json.dumps({k: v for k, v in post_ft_metrics.items() if k not in ("y_true", "y_pred")}, indent=2),
            encoding="utf-8")
        prune_summary.to_csv(pruned_run_dir / "prune_summary.csv", index=False)
        (pruned_run_dir / "pruning_config.json").write_text(json.dumps({
            "base_run_name": config.base_run_name, "prune_ratio": prune_ratio,
            "finetune_epochs": config.finetune_epochs, "finetune_lr": config.finetune_lr,
            "params_before": params_before, "params_after": params_after,
            "param_reduction_pct": 1 - params_after / params_before,
            "pre_finetune_test_f1_macro": pre_ft_metrics["f1_macro"],
            "post_finetune_test_f1_macro": post_ft_metrics["f1_macro"],
        }, indent=2), encoding="utf-8")

        # --- Benchmark CPU memakai harness §9 (reused), dibandingkan terhadap Stage 1 FP32 asli ---
        bench_config = PruningBenchmarkConfig(
            dataset=config.dataset, cpu_threads=config.cpu_threads,
            max_eval_samples=config.max_eval_samples, n_warmup=config.n_warmup, n_repeats=config.n_repeats,
            quality_eval_samples=config.quality_eval_samples, metric_tolerance=config.metric_tolerance,
        )
        bench_results = run_cpu_benchmark_pruned(model_cpu, bench_config, stage1_test_metrics, pruned_run_dir)

        all_results[prune_ratio] = {
            "pruned_run_name": pruned_run_name,
            "params_before": params_before, "params_after": params_after,
            "pre_finetune_f1_macro": pre_ft_metrics["f1_macro"],
            "post_finetune_f1_macro": post_ft_metrics["f1_macro"],
            "post_finetune_qwk": post_ft_metrics["qwk"],
            "benchmark": bench_results,
        }

    return all_results


@dataclass
class PruningBenchmarkConfig:
    '''Config ringan khusus untuk run_cpu_benchmark_pruned -- subset field BenchmarkConfig (§9),
    tanpa run_name/batch_size karena model diberikan langsung sebagai objek, bukan dimuat dari run_name.'''
    dataset: str = "APTOS2019"
    cpu_threads: int = 4
    max_eval_samples: int = 30
    n_warmup: int = 10
    n_repeats: int = 20
    quality_eval_samples: int = 150
    metric_tolerance: float = 0.02


### 10.3 Menjalankan sweep

Jalankan setelah run Stage 1 FP32 (§7) untuk dataset terkait sudah selesai -- `base_run_name` harus persis sama dengan `run_name` Stage 1 yang mau dijadikan basis pruning.


In [51]:
pruning_config = PruningConfig(
    base_run_name="aptos2019_convnextv2tiny_baseline1_fp32_seed42",
    dataset="APTOS2019",
)
pruning_results = run_pruning_sweep(pruning_config)


Fine-tuning recovery akan berjalan di: cuda

########## Pruning ratio: 10% (keep 90% channel ekspansi) ##########
Parameter: 27,870,341 -> 25,272,689 (9.32% reduction)
Sebelum fine-tuning -- test Macro-F1: 0.4113  QWK: 0.7318
  batch 20/92 | running train loss 0.4133
  batch 40/92 | running train loss 0.3376
  batch 60/92 | running train loss 0.2840
  batch 80/92 | running train loss 0.2648
  batch 92/92 | running train loss 0.2561
  [finetune] epoch 1: train_loss=0.2561 val_f1_macro=0.6170 val_qwk=0.8874
  batch 20/92 | running train loss 0.1110
  batch 40/92 | running train loss 0.1017
  batch 60/92 | running train loss 0.1043
  batch 80/92 | running train loss 0.1053
  batch 92/92 | running train loss 0.1060
  [finetune] epoch 2: train_loss=0.1060 val_f1_macro=0.6198 val_qwk=0.8824
  batch 20/92 | running train loss 0.0626
  batch 40/92 | running train loss 0.0617
  batch 60/92 | running train loss 0.0640
  batch 80/92 | running train loss 0.0647
  batch 92/92 | running train loss 0

/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/tmp/ipykernel_1440/44310885.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, str(pat

OpenVINO tidak terpasang; lewati backend ini (opsional menurut brief RQ2).
  timing: 30 sampel x 20 repeats; kualitas: 150 sampel terpisah (satu pass)
          backend  median_latency_ms_per_image  throughput_images_per_second  model_size_mb  f1_macro      qwk
  onnxruntime_cpu                    38.145077                     26.215703      96.518261  0.602615 0.888687
  torchscript_cpu                    52.182662                     19.163453      96.887062  0.602615 0.888687
pytorch_cpu_eager                    59.614457                     16.774455      96.477826  0.602615 0.888687
  Tersimpan di: /content/drive/MyDrive/ConvNeXtV2-DR/checkpoints/aptos2019_convnextv2tiny_baseline1_fp32_seed42_pruned_l1_10pct/cpu_benchmark.json

########## Pruning ratio: 30% (keep 70% channel ekspansi) ##########
Parameter: 27,870,341 -> 20,080,283 (27.95% reduction)
Sebelum fine-tuning -- test Macro-F1: 0.1440  QWK: 0.0095
  batch 20/92 | running train loss 1.1073
  batch 40/92 | running train los

/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/tmp/ipykernel_1440/44310885.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, str(pat

OpenVINO tidak terpasang; lewati backend ini (opsional menurut brief RQ2).
  timing: 30 sampel x 20 repeats; kualitas: 150 sampel terpisah (satu pass)
          backend  median_latency_ms_per_image  throughput_images_per_second  model_size_mb  f1_macro      qwk
  onnxruntime_cpu                    32.345175                     30.916513      76.710770  0.624049 0.897307
  torchscript_cpu                    43.510461                     22.982979      77.079750  0.624049 0.897307
pytorch_cpu_eager                    50.729533                     19.712384      76.670575  0.624049 0.897307
  Tersimpan di: /content/drive/MyDrive/ConvNeXtV2-DR/checkpoints/aptos2019_convnextv2tiny_baseline1_fp32_seed42_pruned_l1_30pct/cpu_benchmark.json

########## Pruning ratio: 50% (keep 50% channel ekspansi) ##########
Parameter: 27,870,341 -> 14,891,333 (46.57% reduction)
Sebelum fine-tuning -- test Macro-F1: 0.1319  QWK: 0.0000
  batch 20/92 | running train loss 1.2178
  batch 40/92 | running train los

/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/tmp/ipykernel_1440/44310885.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, str(pat

OpenVINO tidak terpasang; lewati backend ini (opsional menurut brief RQ2).
  timing: 30 sampel x 20 repeats; kualitas: 150 sampel terpisah (satu pass)
          backend  median_latency_ms_per_image  throughput_images_per_second  model_size_mb  f1_macro      qwk
  onnxruntime_cpu                    24.748325                     40.406776      56.916496   0.40614 0.742931
  torchscript_cpu                    32.883436                     30.410447      57.286293   0.40614 0.742931
pytorch_cpu_eager                    38.944177                     25.677780      56.877545   0.40614 0.742931
  Tersimpan di: /content/drive/MyDrive/ConvNeXtV2-DR/checkpoints/aptos2019_convnextv2tiny_baseline1_fp32_seed42_pruned_l1_50pct/cpu_benchmark.json


In [53]:
import shutil
old_dir = RUN_ROOT / "checkpoints" / "aptos2019_convnextv2tiny_baseline1_fp32_seed42_pruned_l1_50pct"
backup_dir = RUN_ROOT / "checkpoints" / "aptos2019_convnextv2tiny_baseline1_fp32_seed42_pruned_l1_50pct_epochs3"
if old_dir.is_dir() and not backup_dir.is_dir():
    shutil.copytree(old_dir, backup_dir)

In [54]:
pruning_config_50pct_longer = PruningConfig(
    base_run_name="aptos2019_convnextv2tiny_baseline1_fp32_seed42",
    dataset="APTOS2019",
    prune_ratios=(0.50,),       # only re-run the 50% ratio
    finetune_epochs=10,         # was 3 -- see if more recovery training closes the gap
)
pruning_results_50pct_longer = run_pruning_sweep(pruning_config_50pct_longer)

Fine-tuning recovery akan berjalan di: cuda

########## Pruning ratio: 50% (keep 50% channel ekspansi) ##########
Parameter: 27,870,341 -> 14,891,333 (46.57% reduction)
Sebelum fine-tuning -- test Macro-F1: 0.1319  QWK: 0.0000
  batch 20/92 | running train loss 1.2296
  batch 40/92 | running train loss 1.1108
  batch 60/92 | running train loss 1.0226
  batch 80/92 | running train loss 0.9949
  batch 92/92 | running train loss 0.9801
  [finetune] epoch 1: train_loss=0.9801 val_f1_macro=0.3137 val_qwk=0.6895
  batch 20/92 | running train loss 0.8131
  batch 40/92 | running train loss 0.7772
  batch 60/92 | running train loss 0.7600
  batch 80/92 | running train loss 0.7605
  batch 92/92 | running train loss 0.7784
  [finetune] epoch 2: train_loss=0.7784 val_f1_macro=0.3198 val_qwk=0.7439
  batch 20/92 | running train loss 0.6804
  batch 40/92 | running train loss 0.7116
  batch 60/92 | running train loss 0.7160
  batch 80/92 | running train loss 0.7187
  batch 92/92 | running train loss 

/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/tmp/ipykernel_1440/44310885.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy, str(pat

OpenVINO tidak terpasang; lewati backend ini (opsional menurut brief RQ2).
  timing: 30 sampel x 20 repeats; kualitas: 150 sampel terpisah (satu pass)
          backend  median_latency_ms_per_image  throughput_images_per_second  model_size_mb  f1_macro      qwk
  onnxruntime_cpu                    25.377783                     39.404546      56.916496  0.513601 0.816408
  torchscript_cpu                    33.288477                     30.040425      57.286232  0.513601 0.816408
pytorch_cpu_eager                    39.190343                     25.516490      56.877545  0.513601 0.816408
  Tersimpan di: /content/drive/MyDrive/ConvNeXtV2-DR/checkpoints/aptos2019_convnextv2tiny_baseline1_fp32_seed42_pruned_l1_50pct/cpu_benchmark.json


### 10.4 Ringkasan sweep lintas rasio

In [52]:
summary_rows = []
for ratio, r in pruning_results.items():
    best_backend_name, best_backend = min(
        r["benchmark"]["backends"].items(), key=lambda kv: kv[1]["median_latency_ms_per_image"])
    summary_rows.append({
        "prune_ratio": ratio,
        "params_after": r["params_after"],
        "param_reduction_pct": 1 - r["params_after"] / r["params_before"],
        "pre_finetune_f1_macro": r["pre_finetune_f1_macro"],
        "post_finetune_f1_macro": r["post_finetune_f1_macro"],
        "post_finetune_qwk": r["post_finetune_qwk"],
        "fastest_backend": best_backend_name,
        "fastest_latency_ms_per_image": best_backend["median_latency_ms_per_image"],
    })
pruning_summary_df = pd.DataFrame(summary_rows)
pruning_summary_df


,prune_ratio,params_after,param_reduction_pct,pre_finetune_f1_macro,post_finetune_f1_macro,post_finetune_qwk,fastest_backend,fastest_latency_ms_per_image
0,0.1,25272689,0.093205,0.411261,0.596203,0.880176,onnxruntime_cpu,38.145077
1,0.3,20080283,0.279511,0.143984,0.635855,0.882411,onnxruntime_cpu,32.345175
2,0.5,14891333,0.465692,0.131868,0.373404,0.723793,onnxruntime_cpu,24.748325
